In [40]:
import httpx
import time
import pandas as pd
import re
from transformers import pipeline
import numpy as np

In [41]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    multi_label = True
)

Device set to use mps:0


In [42]:
MAX_POSTS = 500

all_post_ids = []

HEADERS = {
    "User-Agent": "mia-natali-research/0.1",
    "Accept": "application/json,text/plain,*/*",
    "Accept-Language": "en-US,en;q=0.9",
}

subreddit = "washingtondc"
search_url = f"https://www.reddit.com/r/{subreddit}/search.json"

params = {
    "q": 'datacenter',
    "restrict_sr": 1
}

env_post_ids = []
env_post_sentiments = []
env_post_sentiment_degrees = []

infr_post_ids = []
infr_post_sentiments = []
infr_post_sentiment_degrees = []

housing_post_ids = []
housing_post_sentiments = []
housing_post_sentiment_degrees = []

econ_post_ids = []
econ_post_sentiments = []
econ_post_sentiment_degrees = []

life_qual_post_ids = []
life_qual_post_sentiments = []
life_qual_post_sentiment_degrees = []

aesth_post_ids = []
aesth_post_sentiments = []
aesth_post_sentiment_degrees = []

gov_post_ids = []
gov_post_sentiments = []
gov_post_sentiment_degrees = []

not_useful_post_ids = []

after = None

themes = ["visual impact of datacenters", "infrastructure and utilities", "housing prices", "economy (jobs)", "quality of life (noise, light pollution)", "environment or climate change", "government (policies, laws)"]
sentiment_labels = ["positive", "negative", "neutral"]

# NEED TO INCLUDE THIS FILTERING
# dont_include = ["server", "rack", "switch", "firewall", "BGP", "latency"]

matched_posts = 0
more_than_one_theme_posts = 0

with httpx.Client(headers=HEADERS, follow_redirects=True) as client:
    while True:
        if after is not None:
            params["after"] = after
        else:
            params.pop("after", None)

        r = client.get(search_url, params=params)
        print("REQUEST URL:", str(r.url))
        r.raise_for_status()
        listing = r.json()

        children = listing["data"]["children"]
        if not children:
            break

        for child in children:
            # data = child.get("data", {})
            # post_id = data.get("id")

            # title = (data.get("title") or "")
            # selftext = (data.get("selftext") or "")
            # text = (title + "\n" + selftext).lower()

            post_id = child["data"]["id"]
            all_post_ids.append(post_id)
            title = (child["data"]["title"] or "")
            selftext = (child["data"]["selftext"] or "")
            text = (title + " " + selftext).lower()

            final_labels = []

            theme_scores = classifier(
                text,
                themes
            )
            
            for i in range(len(theme_scores['labels'])):
                if theme_scores['scores'][i] > 0.5:
                    final_labels.append(theme_scores['labels'][i])
            
            if len(final_labels) > 0:
                matched_posts += 1
            else:
                not_useful_post_ids.append(post_id)
            
            if len(final_labels) > 1:
                more_than_one_theme_posts += 1
            

            for label in final_labels:
                sentiment_results = classifier(
                    text, 
                    sentiment_labels, 
                    hypothesis_template=f"The sentiment towards {label} is {{}}."
                )

                if label == "visual impact of datacenters":
                    aesth_post_ids.append(post_id)
                    aesth_post_sentiments.append(sentiment_results['labels'][0])
                    aesth_post_sentiment_degrees.append(sentiment_results['scores'][0])
                if label == "infrastructure and utilities":
                    infr_post_ids.append(post_id)
                    infr_post_sentiments.append(sentiment_results['labels'][0])
                    infr_post_sentiment_degrees.append(sentiment_results['scores'][0])
                if label == "housing prices":
                    housing_post_ids.append(post_id)
                    housing_post_sentiments.append(sentiment_results['labels'][0])
                    housing_post_sentiment_degrees.append(sentiment_results['scores'][0])
                if label == "economy (jobs)":
                    econ_post_ids.append(post_id)
                    econ_post_sentiments.append(sentiment_results['labels'][0])
                    econ_post_sentiment_degrees.append(sentiment_results['scores'][0])
                if label == "quality of life (noise, light pollution)":
                    life_qual_post_ids.append(post_id)
                    life_qual_post_sentiments.append(sentiment_results['labels'][0])
                    life_qual_post_sentiment_degrees.append(sentiment_results['scores'][0])
                if label == "environment or climate change":
                    env_post_ids.append(post_id)
                    env_post_sentiments.append(sentiment_results['labels'][0])
                    env_post_sentiment_degrees.append(sentiment_results['scores'][0])
                if label == "government (policies, laws)":
                    gov_post_ids.append(post_id)
                    gov_post_sentiments.append(sentiment_results['labels'][0])
                    gov_post_sentiment_degrees.append(sentiment_results['scores'][0])
            
        after = listing["data"]["after"]
        print("After:", after)
        
        if after is None:
            break

        # if len(all_post_ids) >= MAX_POSTS:
        #     break

        time.sleep(0.2)

print("Total posts scanned:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))
print(not_useful_post_ids)

print(gov_post_sentiments)
print(gov_post_sentiment_degrees)

REQUEST URL: https://www.reddit.com/r/washingtondc/search.json?q=datacenter&restrict_sr=1
After: None
Total posts scanned: 5
Total posts with a theme: 4
Found environmental posts: 1
Found infrastructure posts: 1
Found housing posts: 1
Found economic posts: 1
Found life quality posts: 0
Found aesthetic posts: 4
Found government posts: 0
['18dytq']
[]
[]


After categorizing for themes, only 32 out of the 51 retreieved posts remain. However, I believe that this is a good thing because looking through the posts, none of them are revelant. With the other method of categorizing themes, all posts would be used, which I don't believe is good in this scenario.

In [43]:
nova_theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "votes", "number of comments", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

for theme in nova_theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "votes", "number of comments", "environment", "environment sentiment", "environment sentiment degree", "infrastructure", "infrastructure sentiment", "infrastructure sentiment degree", "housing", "housing sentiment", "housing sentiment degree", "economy", "economy sentiment", "economy sentiment degree", "life quality", "life quality sentiment", "life quality sentiment degree", "aesthetics", "aesthetics sentiment", "aesthetics sentiment degree", "government", "government sentiment", "government sentiment degree"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_votes = []
    # What is upvote ratio because for posts with 0 ups and 0 downs I've seen 0.40 something and then for posts with 100 ups and 0 downs ive seen 0.9
    # Should I use the upvotes and downvotes separately or should I use score (upvotes-downvotes) (I'll just use score because it feels like most of the time downs is 0 and I don't know if that's correct)
    post_comment_numbers = []
    with httpx.Client(headers=HEADERS, follow_redirects=True) as client:
        for pid in theme:
            post_url = f"https://www.reddit.com/comments/{pid}.json"                
            r = client.get(post_url)
            r.raise_for_status()
            post_json = r.json()
            text_and_title = post_json[0]["data"]["children"][0]["data"]["title"] + " " + post_json[0]["data"]["children"][0]["data"]["selftext"]
            text_and_title = text_and_title.replace('\n', ' ')
            post_ids.append(pid)
            post_texts.append(text_and_title)
            post_dates.append(post_json[0]["data"]["children"][0]["data"]["created_utc"])
            post_votes.append(post_json[0]["data"]["children"][0]["data"]["score"])
            post_comment_numbers.append(post_json[0]["data"]["children"][0]["data"]["num_comments"])

    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["votes"] = post_votes
    df1["number of comments"] = post_comment_numbers

    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]:
        df1[col] = False

    for col in ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree"]:
        df1[col] = None

    if theme == env_post_ids:
        df1["environment"] = True
        df1["environment sentiment"] = env_post_sentiments
        df1["environment sentiment degree"] = env_post_sentiment_degrees
    if theme == infr_post_ids:
        df1["infrastructure"] = True
        df1["infrastructure sentiment"] = infr_post_sentiments
        df1["infrastructure sentiment degree"] = infr_post_sentiment_degrees
    if theme == housing_post_ids:
        df1["housing"] = True
        df1["housing sentiment"] = housing_post_sentiments
        df1["housing sentiment degree"] = housing_post_sentiment_degrees
    if theme == econ_post_ids:
        df1["economy"] = True
        df1["economy sentiment"] = econ_post_sentiments
        df1["economy sentiment degree"] = econ_post_sentiment_degrees
    if theme == life_qual_post_ids:
        df1["life quality"] = True
        df1["life quality sentiment"] = life_qual_post_sentiments
        df1["life quality sentiment degree"] = life_qual_post_sentiment_degrees
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
        df1["aesthetics sentiment"] = aesth_post_sentiments
        df1["aesthetics sentiment degree"] = aesth_post_sentiment_degrees
    if theme == gov_post_ids:
        df1["government"] = True
        df1["government sentiment"] = gov_post_sentiments
        df1["government sentiment degree"] = gov_post_sentiment_degrees
    posts = pd.concat([posts, df1], ignore_index=True)

/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_1666/2259074375.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_1666/2259074375.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_1666/2259074375.py:68: FutureWarning: The behavior of DataFrame concatenation with 

In [44]:
len(all_post_ids)

5

In [45]:
posts.head(30)

,ids,text,date,votes,number of comments,environment,infrastructure,housing,economy,life quality,...,housing sentiment,housing sentiment degree,economy sentiment,economy sentiment degree,life quality sentiment,life quality sentiment degree,aesthetics sentiment,aesthetics sentiment degree,government sentiment,government sentiment degree
0,59rqcv,How much should I be paid to be critical IT su...,1.477617e+09,13,26,True,True,True,True,False,...,negative,0.933065,positive,0.684188,None,None,None,NaN,None,None
1,59rqcv,How much should I be paid to be critical IT su...,1.477617e+09,12,26,True,True,True,True,False,...,negative,0.933065,positive,0.684188,None,None,None,NaN,None,None
2,59rqcv,How much should I be paid to be critical IT su...,1.477617e+09,11,26,True,True,True,True,False,...,negative,0.933065,positive,0.684188,None,None,None,NaN,None,None
3,59rqcv,How much should I be paid to be critical IT su...,1.477617e+09,11,26,True,True,True,True,False,...,negative,0.933065,positive,0.684188,None,None,None,NaN,None,None
4,aqxuvj,Colo datacenters inside the District of Columbia,1.550246e+09,0,1,False,False,False,False,False,...,None,NaN,None,NaN,None,None,positive,0.624918,None,None
5,1cwtvzo,What is this thing? It’s at the corner of 10th...,1.716248e+09,71,35,False,False,False,False,False,...,None,NaN,None,NaN,None,None,negative,0.250207,None,None
6,27eut2,How much are you paying Comcast for ONLY INTER...,1.402000e+09,11,44,False,False,False,False,False,...,None,NaN,None,NaN,None,None,negative,0.907096,None,None
7,59rqcv,How much should I be paid to be critical IT su...,1.477617e+09,11,26,False,False,False,False,False,...,None,NaN,None,NaN,None,None,negative,0.729692,None,None


In [46]:
def remove_links(text):
    # Regex pattern to find typical URLs (http/https followed by non-whitespace characters)
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    # Replace found URLs with an empty string
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

sample_string = "The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...  [https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#](https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#)  A group of residents from the [Save Bren Mar From Data Center](https://sites.google.com/view/savebrenmar/home) group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial properties.  [https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board](https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board)Fairfax County considers enhancing data center guidelines"
result = remove_links(sample_string)
print(result)

posts["text"] = posts["text"].apply(remove_links)

The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...    A group of residents from the [Save Bren Mar From Data Center] group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial propert

In [47]:
grouping_cols = ["ids", "text", "date"]

theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]
sent_cols  = ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree"]

def first_non_null(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan

agg = {c: "max" for c in theme_cols}              # True if any True
agg.update({c: first_non_null for c in sent_cols}) # keep the real sentiment if present

posts = posts.groupby(grouping_cols, as_index=False).agg(agg)

In [48]:
# datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]
# datacenters_posts = 0

# with httpx.Client(headers=HEADERS, follow_redirects=True) as client:
#         for pid in all_post_ids:
#             post_url = f"https://www.reddit.com/comments/{pid}.json"                
#             r = client.get(post_url)            
#             post_json = r.json()
#             text = post_json[0]["data"]["children"][0]["data"]["selftext"] + post_json[0]["data"]["children"][0]["data"]["title"]
#             if any(kw in text.lower() for kw in datacenters_keywords):
#                 datacenters_posts += 1

# print(datacenters_posts)

In [49]:
datacenters_posts = 0
datacenters_post_texts = []
a=0
invalid_posts = []

for text in posts["text"]:
    if any(kw in text.lower() for kw in datacenters_keywords):
        datacenters_posts += 1
        datacenters_post_texts.append(text)
    else:
        invalid_posts.append(posts.loc[a, "ids"])
    a += 1

print(datacenters_posts)
print(datacenters_post_texts)
print(invalid_posts)

3
['What is this thing? It’s at the corner of 10th and V. There was a QR code leading to this site. https://www.lastenergy.com/?utm_source=signage&amp;utm_medium=banner%20&amp;utm_campaign=datacenter&amp;utm_content=homepage', 'How much should I be paid to be critical IT support in DC? My company recently moved me from our headquarters in Atlanta to our satellite office in DC for a special IT project. I am the only system administrator for this office. Recently one of the applications running out of this office\'s server room became "tier 1 critical".  For [reasons] and ignorance, they won\'t put the application/server in the cloud or a colo datacenter. So I am the only person who can fix it if it breaks to the point where it requires on-hands recovery. (we\'ve got battery backups, vMotion HA, enviro/HVAC monitoring and backup internet circuits and other good policies otherwise)  I was given a 5% raise to move to DC, which I do not think is an adequate COLA for the area. My pay was ok-

Only 21 of the 51 posts that were in the reddit json when you search "datacenter" actually have the word datacenter in the main post, and only 17 out of the 32 that have a useful theme do. Based on the post ids I've looked at, this is because reddit marks a post as containing the search term if any one of the comments contains the search term. However, msny posts whose comments have the word datacenter in them don't actually talk about datacenters, but the comments might be helpful? (Removing links doesn't change the number, so it's all in the text).

Should I throw away these posts and only include posts that actually say datacenters, should I check the upvote ratio for the comments that include datacenters, or something else? I tried just starting with the nova subreddit and filtering from datacenters there instead of having reddit do it for me and it gave me 3 results out of the max 998 posts searched.

Update: When I expand the query to datacenter OR data center OR datacentre OR data centre, I get 131 results but only 3 of them actually have those words in the post, I've searched through some of the posts that come up and they don't even have the keywords in the comments.

In [50]:
print(posts.columns)

Index(['ids', 'text', 'date', 'environment', 'infrastructure', 'housing',
       'economy', 'life quality', 'aesthetics', 'government',
       'environment sentiment', 'environment sentiment degree',
       'infrastructure sentiment', 'infrastructure sentiment degree',
       'housing sentiment', 'housing sentiment degree', 'economy sentiment',
       'economy sentiment degree', 'life quality sentiment',
       'life quality sentiment degree', 'aesthetics sentiment',
       'aesthetics sentiment degree', 'government sentiment',
       'government sentiment degree'],
      dtype='object')


In [51]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    if len(theme_posts) > 0:
        pos = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "positive", f"{theme} sentiment degree"].sum()
        neg = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "negative", f"{theme} sentiment degree"].sum()
        total = theme_posts[f"{theme} sentiment degree"].sum()
        return (pos-neg)/total
    else:
        return None


print("Average sentiment of environmental posts:", avg_sentiment_calculation("environment"))
print("Average sentiment of infrastructural posts:", avg_sentiment_calculation("infrastructure"))
print("Average sentiment of housing-related posts:", avg_sentiment_calculation("housing"))
print("Average sentiment of economic posts:", avg_sentiment_calculation("economy"))
print("Average sentiment of life-quality-related posts:", avg_sentiment_calculation("life quality"))
print("Average sentiment of aesthetics-related posts:", avg_sentiment_calculation("aesthetics"))
print("Average sentiment of governmental posts:", avg_sentiment_calculation("government"))

Average sentiment of environmental posts: -1.0
Average sentiment of infrastructural posts: -1.0
Average sentiment of housing-related posts: -1.0
Average sentiment of economic posts: 1.0
Average sentiment of life-quality-related posts: None
Average sentiment of aesthetics-related posts: -0.5024363778770928
Average sentiment of governmental posts: None
